# EXERCISE SOLUTION

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

In [2]:
# imports

import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import ollama

In [3]:
# Constants

MODEL = "llama3.2"

In [4]:
# A class to represent a Webpage

class Website:
    """
    A utility class to represent a Website that we have scraped
    """
    url: str
    title: str
    text: str

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [5]:
# Let's try one out

ed = Website("https://edwarddonner.com")
print(ed.title)
print(ed.text)

Home - Edward Donner
Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 500,000 enrolled across 194 

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT4o have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [6]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

In [7]:
# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "The contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

## Messages

The API from Ollama expects the same message format as OpenAI:

```
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]

In [8]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

## Time to bring it together - now with Ollama instead of OpenAI

In [9]:
# And now: call the Ollama function instead of OpenAI

def summarize(url):
    website = Website(url)
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']

In [10]:
summarize("https://edwarddonner.com")

"**Summary of Edward Donner's Website**\n=====================================\n\n### Overview\n\nThe website is created by Ed, the co-founder and CTO of Nebula.io. The platform focuses on applying AI to help people discover their potential and pursue their purpose.\n\n### Courses and Resources\n\n*   **AI Curriculum**: A comprehensive curriculum covering various aspects of AI.\n*   **Proficient AI Engineer**: A course series designed to transform learners into proficient AI engineers.\n*   **Outsmart**: An arena where LLMs compete in a battle of diplomacy and deviousness.\n*   **Resources**: Various resources, including Udemy courses and MLOps tracks, are available for learning about AI engineering.\n\n### News and Announcements\n\n*   **February 17, 2026: AI Coder - Vibe Coder to Agentic Engineer**\n*   **January 4, 2026: AI Builder with n8n - Create Agents and Voice Agents**\n*   **September 15, 2025: AI Engineering MLOps Track - Deploy AI to Production**\n*   **May 28, 2025: Which 

In [10]:
# A function to display this nicely in the Jupyter output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [11]:
display_summary("https://edwarddonner.com")

# Summary of Edward Donner's Website

## Overview

Edward Donner is a co-founder and CTO of Nebula.io, applying AI to help people discover their potential. He has experience in LLMs (Large Language Models) and offers resources and courses on AI engineering.

## News/Announcements

* **February 17, 2026**: "AI Coder: Vibe Coder to Agentic Engineer – RESOURCES" - a new resource released
* **January 4, 2026**: "AI Builder with n8n – Create Agents and Voice Agents – RESOURCES" - another new resource released
* **September 15, 2025**: "AI Engineering MLOps Track – Deploy AI to Production – RESOURCES" - a new resource released
* **May 28, 2025**: The order in which to take his AI courses

## About the Website

The website is personal and informal, with Edward sharing his interests in coding, LLMs, and electronic music production. He also provides resources and courses on AI engineering and shares updates about his work at Nebula.io.

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [ ]:
display_summary("https://cnn.com")

In [ ]:
display_summary("https://anthropic.com")

In [12]:
display_summary("https://www.linkedin.com/in/amit-tayal-3b63a6a/")

TypeError: 'NoneType' object is not callable

In [14]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI()
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [15]:
response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

display(response.choices[0].message.content)

'Did you know that there is a species of jellyfish that is immortal? The Turritopsis dohrnii, also known as the "immortal jellyfish," can transform its body into a younger state through a process called transdifferentiation. This means it can essentially revert back to its polyp stage and grow back into an adult again, making it theoretically immune to aging and death! Isn\'t that mind-blowing?'

In [34]:
from selenium import webdriver

def summarize(url):
    website = Website(url)
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']
chrome_options = webdriver.ChromeOptions()

# Uncomment the line below to run Chrome in headless mode
# # chrome_options.add_argument("--headless")
driver = webdriver.Chrome(options=chrome_options)

# # Open Amazon's homepage
driver.get("https://parivesh.nic.in/newupgrade/#/trackYourProposal/")

AdvanceSearch = driver.find_element(By.ID, "button.btn.btn-warning.mn-w-180")
time.sleep(20)
AdvanceSearch.click()

NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":"[id="button.btn.btn-warning.mn-w-180"]"}
  (Session info: chrome=148.0.7778.97); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6d313d6d5+14ef5]
	chromedriver!GetHandleVerifier [0x7ff6d313d740+14f60]
	chromedriver!(No symbol) [0x7ff6d2e8d59d]
	chromedriver!(No symbol) [0x7ff6d2ee6f42]
	chromedriver!(No symbol) [0x7ff6d2ee724c]
	chromedriver!(No symbol) [0x7ff6d2f37967]
	chromedriver!(No symbol) [0x7ff6d2f344ff]
	chromedriver!(No symbol) [0x7ff6d2ed9608]
	chromedriver!(No symbol) [0x7ff6d2eda4f3]
	chromedriver!GetHandleVerifier [0x7ff6d343b779+312f99]
	chromedriver!GetHandleVerifier [0x7ff6d3435c8b+30d4ab]
	chromedriver!GetHandleVerifier [0x7ff6d3459012+330832]
	chromedriver!GetHandleVerifier [0x7ff6d315a485+31ca5]
	chromedriver!GetH

In [ ]:
# Wait for the search bar to be present
search_bar = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.ID, "twotabsearchtextbox"))
)

In [16]:
# Perform a search for "headphones"
search_bar.send_keys("headphones")
  
# Submit the search form (press Enter)
search_bar.submit()

In [17]:
1
2
# Wait for the search results to load
time.sleep(10)

In [18]:
from bs4 import BeautifulSoup
  
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')
products = []
  
# Extract product ASINs
productsHTML = soup.select('div[data-asin]')
for product in productsHTML:
    if product.attrs.get('data-asin'):
        products.append(product.attrs['data-asin'])
  
print(products)

['B0GCJ6D6XR', 'B0FSSBP3BT', 'B0C3HCD34R', 'B0DMP36MVC', 'B083P1HG9S', 'B0G633S569', 'B07ZF8T63K', 'B0FPFSBZBM', 'B09LYF2ST7', 'B0CQXMXJC5', 'B01N6ZJH96', 'B0GLHQLHB2', 'B0BS1RT9S2', 'B09NNBBY8F', 'B07SHBQY7Z', 'B01EF5DBZ6', 'B01EF5DBZ6', 'B0DTRLCM4Q', 'B0GSD4KRC3', 'B0DQ3Y7127', 'B006T9ZKAQ', 'B006T9ZKAQ', 'B0BMLXSNG8', 'B0DQSJ4QHB', 'B0DQSJ4QHB']


In [19]:
# Quit the WebDriver
driver.quit()